# Day 9: Embedding Model Selection, Walked Through

The embedding model is the foundation everything else in RAG builds on. This
notebook compares simulated models of different quality, measures the real cost
tradeoff, looks at domain specialization, surveys real production options, and
shows optimization and fine-tuning techniques.

Run each cell in order. Plain Python throughout, self-contained.

## Cell 1: Why embedding models matter

Every stage of RAG -- retrieval, ranking, generation -- operates on top of
embeddings. If the embeddings are bad, nothing downstream can fully recover. And
unlike most RAG components, swapping embedding models is often one of the
highest-leverage changes you can make to overall quality.

## Cell 2: Compare models on your data

We simulate three model "qualities" as different concept vocabularies -- sparser
vocabulary stands in for a weaker model, richer vocabulary for a stronger one --
then measure retrieval quality with real evaluation metrics, not assertions.

In [ ]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    ma, mb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot / (ma * mb) if ma and mb else 0.0

LOW_QUALITY = {"programming": ["python", "programming"], "retrieval": ["rag", "retrieval"], "search_tech": ["vector", "database", "embeddings"]}
MEDIUM_QUALITY = {
    "programming": ["python", "programming", "language", "syntax", "code"],
    "retrieval": ["rag", "retrieval", "retrieves", "generation", "lookup", "find"],
    "freshness": ["outdated", "fresh", "matters", "old", "problem"],
    "search_tech": ["vector", "database", "embeddings", "similarity", "search", "meaning"],
    "keyword_match": ["keyword", "exact", "words", "matching", "simple", "fast"],
}
HIGH_QUALITY = {
    "programming": ["python", "programming", "language", "syntax", "code", "readable"],
    "retrieval": ["rag", "retrieval", "retrieves", "generation", "lookup", "find", "store"],
    "freshness": ["outdated", "fresh", "matters", "old", "problem", "give", "answers"],
    "search_tech": ["vector", "database", "embeddings", "similarity", "search", "meaning", "lookup", "store", "fast"],
    "keyword_match": ["keyword", "exact", "words", "matching", "simple"],
}

def embed(text, concepts):
    words = set(w.strip("?.,!").lower() for w in text.split())
    return [float(sum(1 for w in words if w in cw)) for cw in concepts.values()]

DOCUMENTS = [
    {"id": 1, "title": "What is Python", "text": "Python is a programming language known for readable syntax and simple code."},
    {"id": 2, "title": "What is RAG", "text": "RAG retrieves relevant documents before generating an answer."},
    {"id": 3, "title": "What is a Vector Database", "text": "A vector database stores data as embeddings and allows fast similarity search."},
    {"id": 5, "title": "Why RAG Matters", "text": "RAG matters because language models can give outdated answers without fresh retrieval."},
    {"id": 6, "title": "Keyword Search Basics", "text": "Keyword search finds exact matching words but misses different phrasing."},
]

def build_retriever(concepts):
    index = [(embed(doc["title"] + " " + doc["text"], concepts), doc) for doc in DOCUMENTS]
    def retrieve(query, k=3):
        q = embed(query, concepts)
        scored = sorted(((doc, cosine_similarity(q, emb)) for emb, doc in index), key=lambda pair: pair[1], reverse=True)
        return [(doc["id"], score) for doc, score in scored[:k] if score > 0]
    return retrieve

query = "How do I store embeddings for fast lookup?"
for name, concepts in [("low", LOW_QUALITY), ("medium", MEDIUM_QUALITY), ("high", HIGH_QUALITY)]:
    results = build_retriever(concepts)(query)
    print(f"{name:8s}: {results}")

The expected answer is doc id 3 (Vector Database). Notice which model quality tiers
actually rank it first -- this is the measurable difference model choice makes.

## Cell 3: Quality vs cost analysis

A real cost calculation for a concrete workload, using approximate real-world
embedding API pricing.

In [ ]:
def workload_cost(num_documents, avg_tokens, num_queries_per_month, price_per_1k_tokens):
    one_time = (num_documents * avg_tokens / 1000) * price_per_1k_tokens
    monthly = (num_queries_per_month * 15 / 1000) * price_per_1k_tokens
    return one_time, monthly

pricing_tiers = {
    "small/fast (e.g. text-embedding-3-small)": 0.00002,
    "large/high-quality (e.g. text-embedding-3-large)": 0.00013,
}

print("Workload: 1,000 documents (200 tokens avg), 10,000 queries/month\n")
for tier, price in pricing_tiers.items():
    one_time, monthly = workload_cost(1000, 200, 10000, price)
    print(f"{tier}: one-time ${one_time:.2f}, then ${monthly:.2f}/month")

## Cell 4: Domain-specific models

A specialized model (richer vocabulary in ONE domain) beats a general-purpose
model within that domain -- but has no advantage outside it.

In [ ]:
GENERAL = {"legal_core": ["agreement", "contract"], "legal_detail": ["liability", "clause"], "code_core": ["function", "variable"]}
LEGAL_SPECIALIZED = {
    "legal_core": ["agreement", "contract"],
    "legal_detail": ["liability", "clause"],
    "legal_remedies": ["indemnify", "termination", "breach", "covenant"],
    "code_core": ["function", "variable"],
}

legal_query = "What happens if a party breaches the agreement and needs to indemnify the other?"
legal_doc = "This contract's termination clause covers breach of covenant and indemnification."

for name, concepts in [("General-purpose", GENERAL), ("Legal-specialized", LEGAL_SPECIALIZED)]:
    sim = cosine_similarity(embed(legal_query, concepts), embed(legal_doc, concepts))
    print(f"{name:20s}: {sim:.3f}")

## Cell 5: Real production models

What the actual code looks like for real embedding providers (no API key needed to
read this).

```python
# OpenAI (hosted API)
from openai import OpenAI
client = OpenAI(api_key="YOUR_API_KEY")
response = client.embeddings.create(model="text-embedding-3-small", input="some text")
embedding = response.data[0].embedding

# Sentence-Transformers (local, open source)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embedding = model.encode("some text")
```

| Model | Hosting | Relative cost | Best for |
|---|---|---|---|
| text-embedding-3-small | API | Very low | Default starting point |
| text-embedding-3-large | API | Moderate | Quality-critical retrieval |
| Sentence-Transformers | Local | Compute only | Privacy, offline, high volume |

## Cell 6: Optimization techniques

Caching and batching are free wins. Quantization and dimensionality reduction trade
a little precision for speed and storage -- worth measuring, not assuming.

In [ ]:
def quantize(embedding, bits=8):
    if not embedding or max(embedding) == min(embedding):
        return list(embedding)
    min_val, max_val = min(embedding), max(embedding)
    levels = 2 ** bits
    step = (max_val - min_val) / (levels - 1)
    return [round((v - min_val) / step) * step + min_val for v in embedding]

full_emb = [0.13, 2.07, 0.13, 1.07, 0.13]
other_emb = [0.07, 1.13, 0.07, 2.13, 0.07]
baseline = cosine_similarity(full_emb, other_emb)

print(f"Full precision: {baseline:.4f}")
for bits in [8, 4, 2]:
    q1, q2 = quantize(full_emb, bits), quantize(other_emb, bits)
    sim = cosine_similarity(q1, q2)
    print(f"{bits}-bit quantized: {sim:.4f} (drift: {abs(sim - baseline):.4f})")

## Cell 7: Local vs API comparison

At low-to-medium volume, API pricing is usually cheaper than owning compute. At
very high volume, or with a pricier high-quality model, self-hosting can win.

In [ ]:
local_monthly_cost = 150.00
api_price = 0.00013  # large/high-quality tier pricing

for label, volume in [("Low (10K/mo)", 10_000), ("Medium (5M/mo)", 5_000_000), ("High (200M/mo)", 200_000_000)]:
    api_cost = (volume * 50 / 1000) * api_price
    cheaper = "API" if api_cost < local_monthly_cost else "Local"
    print(f"{label:16s}: API ${api_cost:>10.2f}/mo vs Local ${local_monthly_cost:.2f}/mo -> cheaper: {cheaper}")

## Cell 8: Key takeaways

- The embedding model is the foundation everything else in RAG builds on -- a bad
  embedding model caps the quality of every later stage.
- Quality, cost, and speed trade off against each other -- there's no universally
  "best" model, only the right model for your constraints.
- Measure model choice on YOUR actual data and queries (Day 8's evaluation
  framework), not by assuming a general benchmark leaderboard transfers directly.
- Domain-specialized models can decisively beat general-purpose ones within their
  specialty, but have no advantage (and sometimes a real disadvantage) outside it.
- Caching and batching are free performance wins with zero quality cost. Quantization
  and dimensionality reduction trade a measurable amount of precision for speed and
  storage savings -- worth it at scale, but verify the quality impact first.
- Local vs. API is a real tradeoff of cost, privacy, and control, not a default
  choice -- compute the actual breakeven for your volume before deciding.
- Fine-tuning is most effective when it targets a SPECIFIC, measured gap (found via
  diagnosis, like Day 8's per_query_diagnosis.py), not as a vague attempt at general
  improvement.